In [3]:
import pandas as pd
import os

folder_path = os.path.join(os.path.dirname(os.getcwd()), 'data', 'processed')

if not os.path.exists(folder_path):
    raise FileNotFoundError(f"Folder not found: {folder_path}")

csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

for file in csv_files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)

    print(f"\nFile: {file}")
    print("Columns:")
    for col in df.columns:
        print(f" - {col}")
    print(f"Number of rows: {len(df)}")



File: olist_geolocation_lookup.csv
Columns:
 - geolocation_zip_code_prefix
 - geolocation_lat
 - geolocation_lng
 - geolocation_city
 - geolocation_state
Number of rows: 19015

File: olist_master_item_level.csv
Columns:
 - order_id
 - customer_id
 - order_status
 - order_purchase_timestamp
 - order_approved_at
 - order_delivered_carrier_date
 - order_delivered_customer_date
 - order_estimated_delivery_date
 - customer_unique_id
 - customer_zip_code_prefix
 - customer_city
 - customer_state
 - order_item_id
 - product_id
 - seller_id
 - shipping_limit_date
 - price
 - freight_value
 - product_category_name
 - product_name_lenght
 - product_description_lenght
 - product_photos_qty
 - product_weight_g
 - product_length_cm
 - product_height_cm
 - product_width_cm
 - product_category_name_english
 - seller_zip_code_prefix
 - seller_city
 - seller_state
 - total_payment_value
 - payment_methods
 - payment_installments
 - review_score
 - review_creation_date
 - delivery_time_days
 - estimate

In [ ]:
import pandas as pd
import numpy as np

# Step 1: Load your two datasets
print("Loading datasets...")
item_df = pd.read_csv(os.path.join(folder_path, "olist_master_item_level.csv"))
order_df = pd.read_csv(os.path.join(folder_path, "olist_master_order_level.csv"))

print(f"Item level rows: {len(item_df)}")
print(f"Order level rows: {len(order_df)}")

# Step 2: Aggregate FROM ITEMS to recreate order-level (what yours should match)
print("\nAggregating item-level to order-level...")
agg = item_df.groupby("order_id").agg({
    "price": "sum",
    "freight_value": "sum", 
    "order_item_id": "count"  # Number of items per order
}).rename(columns={
    "price": "sum_price",
    "freight_value": "sum_freight", 
    "order_item_id": "item_count"
}).reset_index()

print(f"Aggregated to {len(agg)} orders")

# Step 3: Merge with YOUR order-level data
print("\nMerging for validation...")
merged = order_df.merge(agg, on="order_id", how="inner")

# Step 4: Calculate differences (should ALL be zero)
merged["price_diff"] = merged["total_price"] - merged["sum_price"]
merged["freight_diff"] = merged["total_freight"] - merged["sum_freight"]
merged["item_count_diff"] = merged["items_per_order"] - merged["item_count"]

# Step 5: VALIDATION RESULTS
print("\n" + "="*60)
print("DATA INTEGRITY VERIFICATION RESULTS")
print("="*60)

# Summary statistics
diff_cols = ["price_diff", "freight_diff", "item_count_diff"]
summary = merged[diff_cols].describe()

print("\nSummary of Differences (ALL should be 0):")
print(summary.round(2))

# Count non-zero differences (ERRORS)
errors = merged[(merged["price_diff"] != 0) | 
                (merged["freight_diff"] != 0) | 
                (merged["item_count_diff"] != 0)]

print(f"\nERROR COUNT:")
print(f"Orders with price mismatch: {(merged['price_diff'] != 0).sum()}")
print(f"Orders with freight mismatch: {(merged['freight_diff'] != 0).sum()}")
print(f"Orders with item count mismatch: {(merged['item_count_diff'] != 0).sum()}")

# Final verdict
if len(errors) == 0:
    print(" All aggregations match 100%.")
    print("Your PowerBI dashboard will have correct totals.")
else:
    print(f"\n {len(errors)} orders have aggregation errors!")
    print("Fix your groupby/join logic before PowerBI.")
    print("\nSample errors:")
    print(errors[['order_id', 'price_diff', 'freight_diff', 'item_count_diff']].head())

# Save verification report
merged[diff_cols].to_csv(os.path.join(folder_path, "verification_report.csv"), index=False)
print(f"\nVerification report saved: {os.path.join(folder_path, 'verification_report.csv')}")


Loading datasets...
Item level rows: 113727
Order level rows: 99441

Aggregating item-level to order-level...
Aggregated to 99441 orders

Merging for validation...

DATA INTEGRITY VERIFICATION RESULTS

Summary of Differences (ALL should be 0):
       price_diff  freight_diff  item_count_diff
count     99441.0       99441.0          99441.0
mean          0.0           0.0              0.0
std           0.0           0.0              0.0
min          -0.0          -0.0              0.0
25%           0.0           0.0              0.0
50%           0.0           0.0              0.0
75%           0.0           0.0              0.0
max           0.0           0.0              0.0

ERROR COUNT:
Orders with price mismatch: 515
Orders with freight mismatch: 516
Orders with item count mismatch: 0

 968 orders have aggregation errors!
Fix your groupby/join logic before PowerBI.

Sample errors:
                             order_id    price_diff  freight_diff  \
32   00143d0f86d6fbd9f9b38ab440ac